# Label Distribution Visualization

Analyze the distribution and class imbalance of LLM-generated labels from `labels.json` to ensure the dataset is suitable for training a keyword classifier.

**8 output features:**
- `intent` — categorical (informational, commercial, transactional, navigational)
- `entity` — categorical (person, event, location, company, product, other)
- `nsfw` — continuous score [0, 1]
- `saas_potential` — continuous score [0, 1]
- `is_question` — boolean
- `is_comparison` — boolean
- `is_branded` — boolean

In [ ]:
import warnings

import matplotlib.pyplot as plt  # ty:ignore[unresolved-import]
import pandas as pd  # ty:ignore[unresolved-import]
import polars as pl
import seaborn as sns  # ty:ignore[unresolved-import]
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Load data
sample = pl.read_parquet("../../../data/sample.parquet")
raw_labels = pl.read_parquet("../../../data/labels.parquet")

print(f"Sample rows: {len(sample):,}")
print(f"Label entries: {len(raw_labels):,}")
print(f"Sample columns: {sample.columns}")

In [ ]:
## 2. Merge Labels with Sample Data

# Normalize labels JSON — drop empty dicts from failed batches
labels_df = raw_labels.to_pandas()
sample_df = sample.to_pandas()

# Merge on keyword
df = sample_df.merge(labels_df, on="keyword", how="inner")

# Ensure correct dtypes
for col in ["is_question", "is_comparison", "is_branded"]:
    df[col] = df[col].astype(bool)
df["nsfw"] = df["nsfw"].astype(float)
df["saas_potential"] = df["saas_potential"].astype(float)

print(f"Merged shape: {df.shape}")
df.head()

In [ ]:
## 3. Continuous Features: NSFW Score & SaaS Potential

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Continuous Label Distributions", fontsize=14, fontweight="bold")

for ax, col, color in zip(axes, ["nsfw", "saas_potential"], ["#e07070", "#70a0e0"]):
    sns.histplot(df[col], bins=10, kde=True, ax=ax, color=color, edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score (0-1)")
    ax.set_ylabel("Count")

    mean_val = df[col].mean()
    ax.axvline(mean_val, color="white", linestyle="--", linewidth=1.5, label=f"Mean: {mean_val:.2f}")

    # Annotate bins
    near_zero = (df[col] < 0.2).sum()
    near_one = (df[col] > 0.8).sum()
    ax.text(
        0.02,
        0.95,
        f"< 0.2: {near_zero} ({near_zero / len(df) * 100:.1f}%)",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        color="white",
    )
    ax.text(
        0.72,
        0.95,
        f"> 0.8: {near_one} ({near_one / len(df) * 100:.1f}%)",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        color="white",
    )
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
## 4. Categorical Features: Intent & Entity Type

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Categorical Label Distributions", fontsize=14, fontweight="bold")

for ax, col in zip(axes, ["intent", "entity"]):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, palette="muted")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Count")
    ax.set_ylabel("")

    total = len(df)
    for p in ax.patches:
        count = int(p.get_width())
        pct = count / total * 100
        ax.annotate(
            f"{count:,}  ({pct:.1f}%)",
            xy=(p.get_width(), p.get_y() + p.get_height() / 2),
            xytext=(4, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

In [ ]:
## 5. Boolean Features Distribution

bool_cols = ["is_question", "is_comparison", "is_branded"]

# Reshape to long format for faceting
bool_long = df[bool_cols].melt(var_name="feature", value_name="value")
bool_long["value"] = bool_long["value"].map({True: "True", False: "False"})

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
fig.suptitle("Boolean Label Distributions", fontsize=14, fontweight="bold")

total = len(df)
for ax, col in zip(axes, bool_cols):
    counts = df[col].value_counts().reindex([True, False], fill_value=0)
    colors = ["#5ba85a", "#c0504d"]
    bars = ax.bar(
        [f"True\n({counts[True] / total * 100:.1f}%)", f"False\n({counts[False] / total * 100:.1f}%)"],
        [counts[True], counts[False]],
        color=colors,
        edgecolor="white",
        width=0.5,
    )
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2, h + total * 0.005, f"{int(h):,}", ha="center", va="bottom", fontsize=9
        )

    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Count" if ax == axes[0] else "")
    ax.set_ylim(0, total * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
## 6. Correlation Heatmap Between All Label Features

corr_df = df[["intent", "entity", "nsfw", "saas_potential", "is_question", "is_comparison", "is_branded"]].copy()

# Encode categorical columns
for col in ["intent", "entity"]:
    le = LabelEncoder()
    corr_df[col] = le.fit_transform(corr_df[col].astype(str))

# Encode booleans as int
for col in ["is_question", "is_comparison", "is_branded"]:
    corr_df[col] = corr_df[col].astype(int)

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = pd.DataFrame(False, index=corr_matrix.index, columns=corr_matrix.columns)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Correlation Heatmap — Label Features", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
## 7. Class Imbalance Summary Table

rows = []

# Categorical features
for col in ["intent", "entity"]:
    counts = df[col].value_counts()
    majority_cls, majority_n = counts.index[0], counts.iloc[0]
    minority_cls, minority_n = counts.index[-1], counts.iloc[-1]
    rows.append(
        {
            "feature": col,
            "type": "categorical",
            "n_classes": len(counts),
            "majority_class": f"{majority_cls} ({majority_n:,})",
            "minority_class": f"{minority_cls} ({minority_n:,})",
            "imbalance_ratio": round(majority_n / minority_n, 2),
        }
    )

# Continuous features — split at 0.5 threshold
for col in ["nsfw", "saas_potential"]:
    high = (df[col] >= 0.5).sum()
    low = (df[col] < 0.5).sum()
    majority_n, minority_n = max(high, low), min(high, low)
    majority_cls = ">= 0.5" if high >= low else "< 0.5"
    minority_cls = "< 0.5" if high >= low else ">= 0.5"
    rows.append(
        {
            "feature": col,
            "type": "continuous",
            "n_classes": 2,
            "majority_class": f"{majority_cls} ({majority_n:,})",
            "minority_class": f"{minority_cls} ({minority_n:,})",
            "imbalance_ratio": round(majority_n / minority_n, 2) if minority_n > 0 else float("inf"),
        }
    )

# Boolean features
for col in ["is_question", "is_comparison", "is_branded"]:
    counts = df[col].value_counts()
    majority_cls, majority_n = counts.index[0], counts.iloc[0]
    minority_cls, minority_n = counts.index[-1], counts.iloc[-1]
    rows.append(
        {
            "feature": col,
            "type": "boolean",
            "n_classes": 2,
            "majority_class": f"{majority_cls} ({majority_n:,})",
            "minority_class": f"{minority_cls} ({minority_n:,})",
            "imbalance_ratio": round(majority_n / minority_n, 2) if minority_n > 0 else float("inf"),
        }
    )

summary = pd.DataFrame(rows).set_index("feature")


def color_imbalance(val: float) -> str:
    if isinstance(val, float):
        if val > 10:
            return "background-color: #c0504d; color: white"
        elif val > 3:
            return "background-color: #f79646; color: white"
        else:
            return "background-color: #5ba85a; color: white"
    return ""


summary.style.map(color_imbalance, subset=["imbalance_ratio"]).format({"imbalance_ratio": "{:.2f}x"})